In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, avg, col
import os
import shutil
import time
import pandas as pd
from datetime import datetime

spark = SparkSession.builder \
    .appName("Tugas10-StructuredStreaming") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Mengurangi jumlah partisi shuffle default (200) agar pemrosesan pada VM/laptop kita lebih cepat
# -- semakin kecil dataset (seperti simulasi kita), semakin tidak perlu partisi sebanyak itu
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 4.2.0


In [2]:
# Menyiapkan folder kosong yang akan berperan sebagai "sumber data streaming"
folder_stream = "simulasi_tugas10_stream"

shutil.rmtree(folder_stream, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

print(f"Folder simulasi streaming '{folder_stream}' siap (kosong).")

Folder simulasi streaming 'simulasi_tugas10_stream' siap (kosong).


In [4]:
# Mendefinisikan skema secara eksplisit — wajib untuk streaming source berbasis berkas
skema_transaksi = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

print("Skema berhasil didefinisikan:")
print(skema_transaksi)

Skema berhasil didefinisikan:
StructType([StructField('pesanan_id', StringType(), True), StructField('nama_cabang', StringType(), True), StructField('nama_menu', StringType(), True), StructField('qty', IntegerType(), True), StructField('waktu_pesan', TimestampType(), True)])


In [5]:
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_transaksi) \
    .option("header", True) \
    .load(folder_stream)

print("Streaming DataFrame berhasil didefinisikan (belum berjalan — baru 'rencana').")
print("df_stream.isStreaming:", df_stream.isStreaming)

Streaming DataFrame berhasil didefinisikan (belum berjalan — baru 'rencana').
df_stream.isStreaming: True


In [7]:
# perbarui skema untuk menyertakan 'harga_satuan' (tipe IntegerType atau DoubleType)
skema_transaksi = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True), # <-- Ditambahkan di sini
    StructField("waktu_pesan", TimestampType(), True),
])

# baca ulang streaming DataFrame dengan skema yang baru
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_transaksi) \
    .option("header", True) \
    .load(folder_stream)

# tambahkan kolom turunan 'total_harga' (qty x harga_satuan)
df_stream_dengan_total = df_stream.withColumn(
    "total_harga", 
    col("qty") * col("harga_satuan")
)

# bat agregasi groupBy("nama_cabang") 
# Menghitung total_pendapatan (dari total_harga) dan jumlah pesanan (count)
agregasi_stream = df_stream_dengan_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# jalankan sebagai streaming query dengan sink memory
query = agregasi_stream.writeStream \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .outputMode("complete") \
    .start()

print("Streaming query 'agregasi_cabang' berhasil dijalankan!")

Streaming query 'agregasi_cabang' berhasil dijalankan!


In [8]:
agregasi_waktu_stream = df_stream_dengan_total.groupBy(
    window(col("waktu_pesan"), "2 minutes"),
    col("nama_cabang")
).agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query_waktu = agregasi_waktu_stream.writeStream \
    .outputMode("update") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming query 'agregasi_waktu' dimulai! Status:", query_waktu.status)

Streaming query 'agregasi_waktu' dimulai! Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [10]:
folder_stream = "simulasi_tugas10_stream"

# --- BATCH 1 ---
pd.DataFrame({
    "pesanan_id": ["P001", "P002", "P003"],
    "nama_cabang": ["Cabang Magelang", "Cabang Solo", "Cabang Magelang"],
    "nama_menu": ["Nasi Goreng", "Mie Goreng", "Es Teh"],
    "qty": [2, 1, 3],
    "harga_satuan": [15000, 12000, 5000],
    "waktu_pesan": ["2026-11-11 20:00:00", "2026-11-11 20:00:30", "2026-11-11 20:01:00"]
}).to_csv(f"{folder_stream}/batch_1.csv", index=False)

print("Batch 1 dikirim. Menunggu Spark memproses...")
time.sleep(6)  # memberi waktu Spark memproses micro-batch (proses pertama kali butuh waktu ekstra)

print("\nHasil Agregasi Cabang (Batch 1):")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

print("\nHasil Agregasi Waktu (Batch 1):")
spark.sql("SELECT window.start, window.end, nama_cabang, total_pendapatan, jumlah_pesanan FROM agregasi_waktu ORDER BY window.start").show(truncate=False)


# --- BATCH 2 ---
pd.DataFrame({
    "pesanan_id": ["P004", "P005"],
    "nama_cabang": ["Cabang Solo", "Cabang Semarang"],
    "nama_menu": ["Ayam Penyet", "Bakso"],
    "qty": [2, 1],
    "harga_satuan": [20000, 18000],
    "waktu_pesan": ["2026-11-11 20:01:30", "2026-11-11 20:02:10"]
}).to_csv(f"{folder_stream}/batch_2.csv", index=False)

print("\nBatch 2 dikirim. Menunggu Spark memproses...")
time.sleep(6)

print("\nHasil Agregasi Cabang (Batch 2 Terakumulasi):")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

print("\nHasil Agregasi Waktu (Batch 2):")
spark.sql("SELECT window.start, window.end, nama_cabang, total_pendapatan, jumlah_pesanan FROM agregasi_waktu ORDER BY window.start").show(truncate=False)


# --- BATCH 3 ---
pd.DataFrame({
    "pesanan_id": ["P006", "P007"],
    "nama_cabang": ["Cabang Magelang", "Cabang Semarang"],
    "nama_menu": ["Es Jeruk", "Mie Ayam"],
    "qty": [4, 2],
    "harga_satuan": [6000, 15000],
    "waktu_pesan": ["2026-11-11 20:02:45", "2026-11-11 20:03:15"]
}).to_csv(f"{folder_stream}/batch_3.csv", index=False)

print("\nBatch 3 dikirim. Menunggu Spark memproses...")
time.sleep(6)

print("\nHasil Agregasi Cabang (Batch 3 Final Terakumulasi):")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

print("\nHasil Agregasi Waktu (Batch 3 Final):")
spark.sql("SELECT window.start, window.end, nama_cabang, total_pendapatan, jumlah_pesanan FROM agregasi_waktu ORDER BY window.start").show(truncate=False)

Batch 1 dikirim. Menunggu Spark memproses...

Hasil Agregasi Cabang (Batch 1):
+---------------+----------------+--------------+
|    nama_cabang|total_pendapatan|jumlah_pesanan|
+---------------+----------------+--------------+
|Cabang Magelang|           45000|             2|
|    Cabang Solo|           12000|             1|
+---------------+----------------+--------------+


Hasil Agregasi Waktu (Batch 1):
+-------------------+-------------------+---------------+----------------+--------------+
|start              |end                |nama_cabang    |total_pendapatan|jumlah_pesanan|
+-------------------+-------------------+---------------+----------------+--------------+
|2026-11-11 20:00:00|2026-11-11 20:02:00|Cabang Solo    |12000           |1             |
|2026-11-11 20:00:00|2026-11-11 20:02:00|Cabang Magelang|45000           |2             |
+-------------------+-------------------+---------------+----------------+--------------+


Batch 2 dikirim. Menunggu Spark memproses...


In [15]:
# 1. Bersihkan folder output dan checkpoint sebelumnya jika ada
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)

# 2. Membuat streaming query ketiga (File Sink - Parquet) tanpa agregasi
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Streaming query ketiga (Parquet sink) dimulai! Status:", query3.status)

# 3. Kirim satu batch data baru untuk memicu file sink
waktu_sekarang = datetime.now()
pd.DataFrame({
    "pesanan_id": ["P999", "P998"],
    "nama_cabang": ["Cabang Magelang", "Cabang Solo"],
    "nama_menu": ["Es Degan", "Jus Alpukat"],
    "qty": [2, 1],
    "harga_satuan": [8000, 10000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 2,
}).to_csv(f"{folder_stream}/batch_parquet_test.csv", index=False)

print("Batch baru dikirim. Menunggu Spark menulis ke sink Parquet...")
time.sleep(6)  # Beri waktu Spark memproses micro-batch dan menulis ke disk

# 4. Membuktikan berkas Parquet benar-benar tersimpan di folder
print("\n--- Isi Folder tugas10_output ---")
!ls -lh tugas10_output

# 5. Membaca kembali berkas Parquet sebagai DataFrame batch biasa
df_hasil_parquet = spark.read.parquet("tugas10_output")

print("\n--- Data yang Berhasil Dibaca Kembali dari Format Parquet ---")
df_hasil_parquet.show(truncate=False)

Streaming query ketiga (Parquet sink) dimulai! Status: {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}
Batch baru dikirim. Menunggu Spark menulis ke sink Parquet...


26/11/11 20:48:36 ERROR MicroBatchExecution: Query [id = 0413ab7f-b60a-44ca-938c-c7daade49781, runId = 0734129b-99ec-4bbb-a510-d18e2080d1b9] terminated with error
org.apache.spark.SparkConcurrentModificationException: Multiple streaming queries are concurrently using file:/home/imam/praktikum-bigdata/tugas10_checkpoint/sources/0.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.multiStreamingQueriesUsingPathConcurrentlyError(QueryExecutionErrors.scala:1834)
	at org.apache.spark.sql.execution.streaming.checkpointing.HDFSMetadataLog.write(HDFSMetadataLog.scala:217)
	at org.apache.spark.sql.execution.streaming.checkpointing.HDFSMetadataLog.addNewBatchByStream(HDFSMetadataLog.scala:244)
	at org.apache.spark.sql.execution.streaming.checkpointing.HDFSMetadataLog.add(HDFSMetadataLog.scala:137)
	at org.apache.spark.sql.execution.streaming.runtime.CompactibleFileStreamLog.add(CompactibleFileStreamLog.scala:172)
	at org.apache.spark.sql.execution.streaming.runtime.FileStreamSourceLog.add(Fi


--- Isi Folder tugas10_output ---
total 24K
-rw-r--r-- 1 imam imam 1.9K Nov 11 20:48 part-00000-36795c90-ccb7-4a57-9860-b5c5842c69ee-c000.snappy.parquet
-rw-r--r-- 1 imam imam 1.9K Nov 11 20:48 part-00000-daf7e367-d2b6-4c9b-90eb-cb28d4fe1311-c000.snappy.parquet
-rw-r--r-- 1 imam imam 1.8K Nov 11 20:48 part-00001-47f404ba-db1f-45f1-86fe-307bcccc81aa-c000.snappy.parquet
-rw-r--r-- 1 imam imam 1.9K Nov 11 20:48 part-00002-13f3a7fc-3d77-4126-8e54-31e4f2fa2370-c000.snappy.parquet
-rw-r--r-- 1 imam imam 1.8K Nov 11 20:48 part-00003-a7743382-af45-4137-86de-cc4d1a01b9c2-c000.snappy.parquet
drwxr-xr-x 2 imam imam 4.0K Nov 11 20:48 _spark_metadata

--- Data yang Berhasil Dibaca Kembali dari Format Parquet ---
+----------+---------------+-----------+---+------------+-------------------+
|pesanan_id|nama_cabang    |nama_menu  |qty|harga_satuan|waktu_pesan        |
+----------+---------------+-----------+---+------------+-------------------+
|P001      |Cabang Magelang|Nasi Goreng|2  |15000       